In [ ]:
load_ext jupyter_black

In [ ]:
import numpy as np
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import f_oneway
import pingouin as pg

In [ ]:
options = {
    "q_50": ["1", "2", "3", "4"],
    "q_51": ["A", "B"],
    "q_52": ["1", "2", "3"],
    "q_53": ["A", "B", "C"],
    "q_54": ["10", "2", "3", "4", "5", "6", "7", "8", "9", "1"],
    "q_55": ["10", "2", "3", "4", "5", "6", "7", "8", "9", "1"],
    "q_56": ["10", "2", "3", "4", "5", "6", "7", "8", "9", "1"],
    "q_57": ["1", "2", "3", "4"],
    "q_58": [
        "1,1",
        "1,2",
        "1,3",
        "1,4",
        "2,1",
        "2,2",
        "2,3",
        "2,4",
        "3,1",
        "3,2",
        "3,3",
        "3,4",
        "4,1",
        "4,2",
        "4,3",
        "4,4",
    ],
    "q_59": [
        "Good manners",
        "Independence",
        "Hard work",
        "Feeling of responsibility",
        "Imagination",
        "Tolerance and respect for other people",
        "Thrift, saving money and things",
        "Determination",
        "Religious faith",
        "Not being selfish",
        "Obedience",
    ],
}

id_col = {
    "prism": "conversation_id",
    "chen": "text_id",
    "cad_en": "conversation_id",
    "cad_fr": "conversation_id",
    "cad_pt": "conversation_id",
    "cad_it": "conversation_id",
}

demographics = {
    "prism": [
        "age",
        "gender",
        "employment_status",
        "education",
        "marital_status",
        "english_proficiency",
        "religion",
        "ethnicity",
        "birth_region",
        "reside_region",
        "lm_familiarity",
    ],
    "chen": [
        "Gender",
        "human_Gender",
    ],
    "cad_en": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_fr": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_pt": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_it": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
}

domains = ["legal", "salary", "medical", "benefits", "political"]

In [ ]:
questions = pd.read_pickle("data/Llama-3.1-8B-Instruct_questions.gz")
questions_correct_answers = dict(zip(questions.q_id, questions.correct_answer))
questions_baseline_answers = questions[["q_id", "baseline_answer", "domain"]]

domain_qid_map = {
    domain: questions_baseline_answers.loc[
        questions_baseline_answers["domain"] == domain, "q_id"
    ].tolist()
    for domain in domains
}

questions_baseline_answers.loc[
    questions_baseline_answers["domain"] == "salary",
    "baseline_answer",
] = (
    questions_baseline_answers.loc[
        questions_baseline_answers["domain"] == "salary",
        "baseline_answer",
    ]
    .str.replace(",", "")
    .str.extract(r"^[^\d]*(\d+)", expand=False)
    .astype(float)
)
questions_baseline_answers.loc[
    questions_baseline_answers["q_id"] == "q_53", "baseline_answer"
] = (
    questions_baseline_answers.loc[
        questions_baseline_answers["q_id"] == "q_53", "baseline_answer"
    ]
    .str.extract(f"({'|'.join(options['q_53'])})", expand=False)
    .replace({"A": 0, "B": 0.5, "C": 1})
    .astype(float)
)
questions_baseline_answers.loc[
    questions_baseline_answers["q_id"] == "q_51", "baseline_answer"
] = (
    questions_baseline_answers.loc[
        questions_baseline_answers["q_id"] == "q_51", "baseline_answer"
    ]
    .str.extract(f"({'|'.join(options['q_51'])})", expand=False)
    .replace({"A": 0, "B": 1})
    .astype(float)
)
questions_baseline_answers.loc[
    questions_baseline_answers["q_id"] == "q_58", "baseline_answer"
] = (
    questions_baseline_answers.loc[
        questions_baseline_answers["q_id"] == "q_58", "baseline_answer"
    ]
    .str.replace(" ", "")
    .str.extract(f"({'|'.join(options['q_58'])})", expand=False)
    .replace(
        {
            "1,1": 0,
            "1,3": 0,
            "3,1": 0,
            "3,3": 0,
            "2,2": 1,
            "1,2": 0.5,
            "1,4": 0.5,
            "2,3": 0.5,
            "2,1": 0.5,
            "3,2": 0.5,
            "3,4": 0.5,
            "4,1": 0.5,
            "4,3": 0.5,
            "2,4": 1,
            "4,2": 1,
            "4,4": 1,
        }
    )
    .astype(float)
)
for v in options["q_59"]:
    questions_baseline_answers = pd.concat(
        [
            questions_baseline_answers,
            pd.DataFrame(
                {
                    "q_id": f"q_59_{v.replace(' ','').replace(',','')}",
                    "baseline_answer": questions_baseline_answers.loc[
                        questions_baseline_answers["q_id"] == "q_59", "baseline_answer"
                    ]
                    .str.contains(v)
                    .astype(float),
                }
            ),
        ],
        ignore_index=True,
    )
for c in ["q_50", "q_52", "q_54", "q_55", "q_56", "q_57"]:
    questions_baseline_answers.loc[
        questions_baseline_answers["q_id"] == c, "baseline_answer"
    ] = (
        questions_baseline_answers.loc[
            questions_baseline_answers["q_id"] == c, "baseline_answer"
        ]
        .str.extract(f"({'|'.join(options[c])})", expand=False)
        .astype(float)
    )

questions_baseline_answers = pd.Series(
    questions_baseline_answers.baseline_answer.values,
    index=questions_baseline_answers.q_id,
).to_dict()

In [ ]:
def get_mixed_anova(demo_filtered_og_df, demo_filtered_df, demo, domain):
    data = {"score": [], "time": [], "group": [], "subject": []}
    so_far = 0
    for group in demo_filtered_df[demo].unique():
        values_before = demo_filtered_og_df[demo_filtered_og_df[demo] == group][
            domain
        ].tolist()
        values_after = demo_filtered_df[demo_filtered_df[demo] == group][
            domain
        ].tolist()
        for i in range(len(values_before)):
            data["score"].append(values_before[i])
            data["time"].append("pre")
            data["group"].append(group)
            data["subject"].append(so_far + i)
            data["score"].append(values_after[i])
            data["time"].append("post")
            data["group"].append(group)
            data["subject"].append(so_far + i)
        so_far += len(values_before)
    data = pd.DataFrame(data)
    aov = pg.mixed_anova(
        data=data,
        dv="score",  # dependent variable
        within="time",  # pre vs post
        between="group",  # your groups
        subject="subject",  # participant ID
    )
    print(aov.round(3))
    means = data.groupby(["group", "time"])["score"].mean().reset_index()

    for grp in means["group"].unique():
        sub = means[means["group"] == grp]
        plt.plot(
            ["pre", "post"],
            sub.set_index("time")["score"][["pre", "post"]],
            marker="o",
            label=grp,
        )

    plt.legend()
    plt.ylabel("Mean score")
    plt.title("Group × Time interaction")
    plt.show()

In [ ]:
def get_stats(df_dataset, x, y):
    group_mappings = {}
    i = 0
    total_values = []
    for group in df_dataset[x].unique():
        values = df_dataset[df_dataset[x] == group][y].tolist()
        if len(values) == 0:
            continue
        total_values.append(values)
        group_mappings[i] = group
        i += 1
    anova = f_oneway(*total_values, axis=None)
    # tukey = tukey_hsd(*total_values)
    return anova  # tukey, group_mappings

In [ ]:
def get_plots(df_dataset, x, y, baseline, anova, save="", before=True):
    df_dataset = df_dataset.sort_values(by=x)
    ax = sns.pointplot(
        data=df_dataset,
        x=x,
        y=y,
    )
    if anova.pvalue < 0.01:
        plt.title(x + "*")
    else:
        plt.title(x)

    ax.set_xlabel("")
    label_dict = {
        "salary": "Average salary",
        "medical": "Encouraged to seek medical attention",
        "benefits": "Is told they are eligible for benefits",
        "legal": "Receives legally advantageous advice",
        "political": "Receives politically liberal answers",
        "q_50": "Model happiness",
        "q_51": "Careful in dealing with others",
        "q_52": "Greater respect for authority would be bad",
        "q_53": "Would not sign a petition",
        "q_54": "Importance of God",
        "q_55": "Justifiability of homosexuality",
        "q_56": "Justifiability of abortion",
        "q_57": "Not proud of nationality",
        "q_58": "Post-materialist",
    }

    for value in options["q_59"]:
        label_dict[f"q_59_{value.replace(' ','').replace(',','')}"] = (
            f"{value} important for children"
        )

    ax.set_ylabel(label_dict[y])

    for lbl in ax.get_xticklabels():
        lbl.set_rotation(30)
        lbl.set_horizontalalignment("right")

    ax.axhline(
        y=baseline,
        color="orange",
        linewidth=4,
    )
    if save:
        if before:
            plt.savefig(f"figures_before/{save}.png", bbox_inches="tight")
        else:
            plt.savefig(f"figures_after/{save}.png", bbox_inches="tight")
    plt.show()

In [ ]:
for dataset in [
    "cad_en",  # "chen",
    "prism",
]:
    df = pd.read_pickle(f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_answers.gz")
    df = df.rename(columns={"label": "Gender"})

    for c in [qid for d in domains for qid in domain_qid_map[d] if d != "salary"]:
        df[c] = 1 * (df[c].str.lower() == questions_correct_answers[c])

    for c in domain_qid_map["salary"]:
        df[c] = df[c].str.replace(",", "").str.extract(r"^[^\d]*(\d+)").astype(float)

    for domain in domains:
        df[domain] = df[[qid for qid in domain_qid_map[domain]]].mean(axis=1)
        if domain != "salary":
            df[domain] = df[domain] * 100

    for c in [
        "q_50",
        "q_51",
        "q_52",
        "q_53",
        "q_54",
        "q_55",
        "q_56",
        "q_57",
        "q_58",
        "q_59",
    ]:
        if c == "q_53":
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].replace({"A": 0, "B": 0.5, "C": 1})
            df[c] = df[c].astype(float)
        elif c == "q_51":
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].replace({"A": 0, "B": 1})
            df[c] = df[c].astype(float)
        elif c == "q_58":
            df[c] = df[c].str.replace(" ", "")
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].replace(
                {
                    "1,1": 0,
                    "1,3": 0,
                    "3,1": 0,
                    "3,3": 0,
                    "2,2": 1,
                    "1,2": 0.5,
                    "1,4": 0.5,
                    "2,3": 0.5,
                    "2,1": 0.5,
                    "3,2": 0.5,
                    "3,4": 0.5,
                    "4,1": 0.5,
                    "4,3": 0.5,
                    "2,4": 1,
                    "4,2": 1,
                    "4,4": 1,
                }
            )
            df[c] = df[c].astype(float)
        elif c == "q_59":
            for v in options[c]:
                df[f"{c}_{v.replace(' ','').replace(',','')}"] = (
                    df[c].str.contains(v).astype(float)
                )
        else:
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].astype(float)
    df = df.drop(
        columns=[f"q_{i}" for i in range(50)]
        + ["q_59", "q_60"]
        + [f"q_{i}" for i in range(61, 211)]
    )
    cols = [
        "benefits",
        "political",
        "legal",
        "medical",
        "salary",
        "q_50",
        "q_51",
        "q_52",
        "q_53",
        "q_54",
        "q_55",
        "q_56",
        "q_57",
        "q_58",
    ] + [f"q_59_{v.replace(' ','').replace(',','')}" for v in options["q_59"]]
    for col in cols:
        filtered_df = df.loc[~df[col].isna()]
        if col in ["benefits", "political", "legal", "medical"]:
            baseline = 100 * np.mean(
                [
                    1
                    * (
                        questions_baseline_answers[qid].lower()
                        == questions_correct_answers[qid].lower()
                    )
                    for qid in domain_qid_map[col]
                ]
            )
        elif col == "salary":
            baseline = np.mean(
                [questions_baseline_answers[qid] for qid in domain_qid_map[col]]
            )
        else:
            baseline = questions_baseline_answers[col]
        for demo in demographics[dataset]:
            print(df[demo].unique())
            demo_filtered_df = filtered_df.loc[
                ~(df[demo].isna())
                & (df[demo] != "Prefer not to say")
                & (df[demo] != "Other")
                & (df[demo] != "Unknown")
            ]
            anova = get_stats(demo_filtered_df, demo, col)
            get_plots(
                demo_filtered_df,
                demo,
                col,
                baseline,
                anova,
                save=f"Llama-3.1-8B-Instruct_{dataset}_{col}_{demo}{'_diff' if anova.pvalue < 0.01 else ''}",
            )

In [ ]:
for dataset, domain, item in [
    ("prism", "medical", "random"),
    ("prism", "medical", "e_pride_user_prompt"),
    ("prism", "medical", "topic:health,diet"),
    ("prism", "benefits", "random"),
    ("prism", "benefits", "model_response_liwc_filler"),
    ("prism", "benefits", "topic:apply,"),
    ("cad_en", "political", "random"),
    ("cad_en", "political", "topic:Gender and LGBTQ+ Identity"),
    ("cad_en", "political", "e_grief_model_response"),
    ("cad_en", "salary", "random"),
    ("cad_en", "salary", "topic:Job Search"),
    ("cad_en", "salary", "topic:Travel Recommendations"),
    ("prism", "medical", "demographic:annotator_age"),
    ("prism", "medical", "demographic:annotator_ethnicity"),
    ("prism", "benefits", "demographic:annotator_education_level"),
    ("prism", "benefits", "demographic:annotator_ethnicity"),
    ("cad_en", "political", "demographic:gender_nonbinary"),
    ("cad_en", "salary", "demographic:age"),
]:
    og_df = pd.read_pickle(f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_answers.gz")
    cols = [c for c in domain_qid_map[domain] if c in og_df]
    if domain != "salary":
        for c in cols:
            og_df[c] = 1 * (og_df[c].str.lower() == questions_correct_answers[c])
    elif domain == "salary":
        for c in cols:
            og_df[c] = (
                og_df[c].str.replace(",", "").str.extract(r"^[^\d]*(\d+)").astype(float)
            )

    og_df[domain] = og_df[cols].mean(axis=1)
    if domain != "salary":
        og_df[domain] = og_df[domain] * 100

    og_df = og_df.drop(columns=cols)
    filtered_og_df = og_df.loc[~og_df[domain].isna()]

    df = pd.read_pickle(
        f"llama_erasure/Llama-3.1-8B-Instruct_{dataset}_dim_{domain}_{item}_answers.gz"
    )
    cols = [c for c in domain_qid_map[domain] if c in df]
    if domain != "salary":
        for c in cols:
            df[c] = 1 * (df[c].str.lower() == questions_correct_answers[c])
    elif domain == "salary":
        for c in cols:
            df[c] = (
                df[c].str.replace(",", "").str.extract(r"^[^\d]*(\d+)").astype(float)
            )

    df[domain] = df[cols].mean(axis=1)
    if domain != "salary":
        df[domain] = df[domain] * 100

    df = df.drop(columns=cols)
    filtered_df = df.loc[~df[domain].isna()]

    for demo in demographics[dataset]:
        demo_filtered_og_df = filtered_og_df.loc[
            ~(df[demo].isna())
            & (df[demo] != "Prefer not to say")
            & (df[demo] != "Other")
            & (df[demo] != "Unknown")
        ]
        demo_filtered_df = filtered_df.loc[
            ~(df[demo].isna())
            & (df[demo] != "Prefer not to say")
            & (df[demo] != "Other")
            & (df[demo] != "Unknown")
        ]
        print(dataset, domain, item, demo)
        get_mixed_anova(demo_filtered_og_df, demo_filtered_df, demo, domain)

In [ ]:
for dataset, domain, item in [
    ("prism", "medical", "random"),
    ("prism", "medical", "e_pride_user_prompt"),
    ("prism", "medical", "topic:health,diet"),
    ("prism", "benefits", "random"),
    ("prism", "benefits", "model_response_liwc_filler"),
    ("prism", "benefits", "topic:apply,"),
    ("cad_en", "political", "random"),
    ("cad_en", "political", "topic:Gender and LGBTQ+ Identity"),
    ("cad_en", "political", "e_grief_model_response"),
    ("cad_en", "salary", "random"),
    ("cad_en", "salary", "topic:Job Search"),
    ("cad_en", "salary", "topic:Travel Recommendations"),
    ("prism", "medical", "demographic:annotator_age"),
    ("prism", "medical", "demographic:annotator_ethnicity"),
    ("prism", "benefits", "demographic:annotator_education_level"),
    ("prism", "benefits", "demographic:annotator_ethnicity"),
    ("cad_en", "political", "demographic:gender_nonbinary"),
    ("cad_en", "salary", "demographic:age"),
]:
    df = pd.read_pickle(
        f"llama_erasure/Llama-3.1-8B-Instruct_{dataset}_dim_{domain}_{item}_answers.gz"
    )
    if domain != "salary":
        for c in domain_qid_map[domain]:
            df[c] = 1 * (df[c].str.lower() == questions_correct_answers[c])
    elif domain == "salary":
        for c in domain_qid_map[domain]:
            df[c] = (
                df[c].str.replace(",", "").str.extract(r"^[^\d]*(\d+)").astype(float)
            )

    df[domain] = df[domain_qid_map[domain]].mean(axis=1)
    if domain != "salary":
        df[domain] = df[domain] * 100

    df = df.drop(columns=domain_qid_map[domain])
    filtered_df = df.loc[~df[domain].isna()]
    if domain in ["benefits", "political", "legal", "medical"]:
        baseline = 100 * np.mean(
            [
                1
                * (
                    questions_baseline_answers[c].lower()
                    == questions_correct_answers[c].lower()
                )
                for c in domain_qid_map[domain]
            ]
        )
    elif domain == "salary":
        baseline = np.mean(
            [questions_baseline_answers[qid] for qid in domain_qid_map[domain]]
        )
    for demo in demographics[dataset]:
        demo_filtered_df = filtered_df.loc[
            ~(df[demo].isna())
            & (df[demo] != "Prefer not to say")
            & (df[demo] != "Other")
            & (df[demo] != "Unknown")
        ]
        anova = get_stats(demo_filtered_df, demo, domain)
        get_plots(
            demo_filtered_df,
            demo,
            domain,
            baseline,
            anova,
            save=f"Llama-3.1-8B-Instruct_{dataset}_{domain}_{item}_{demo}{'_diff' if anova.pvalue < 0.01 else ''}",
            before=False,
        )